## Code to generate the m-climate for different variables

In [1]:
# imports
import xarray as xr
import numpy as np
import pandas as pd
import glob



In [13]:
input_path = "C:\\Users\\Home\\Documents\\KIT\\Hiwi\\ECMWF_Download\\wind_700_500\\*.nc"
output_path = "E:\\Hiwi\\Github_Kenya\\ECMWF-S2S4AFRICA\\m-climate\\700_500_wind\\m-climate_"

for path in glob.glob(input_path):
    data = xr.open_dataset(path)

    # sample data[step] down to weekly values
    data = data.resample(step = "7D").mean()

    # change step into datetime format
    data["step"] = data["time"].values[-1] + data["step"] #- pd.Timedelta("6h")

    # stack number and time dimension
    data = data.stack({"numbertime" : ("number","time")})

    # compute quantiles
    quantiles = np.linspace(0,1,101)
    data = data.quantile(quantiles,dim="numbertime",method="linear")

    # rename step dim into time dim
    data = data.rename({"step":"time"})

    data.to_netcdf(f"{output_path}{path[-13:]}")
    

In [ ]:
# Tmin_Tmax additionally shift the time back by 6 hours in order to be able to aggregate onto the correct date

input_path = "E:\\Hiwi\\ECMWF_Download\\T_min_max_6h\\*.nc"
output_path = "E:\\Hiwi\\Github_Kenya\\ECMWF-S2S4AFRICA\\m-climate\\Tmin_Tmax\\m-climate_"

for path in glob.glob(input_path):
    data = xr.open_dataset(path)

    # sample data[step] down to weekly values
    data = data.resample(step = "7D").mean()

    # change step into datetime format
    data["step"] = data["time"].values[-1] + data["step"] - pd.Timedelta("6h")

    # stack number and time dimension
    data = data.stack({"numbertime" : ("number","time")})

    # compute quantiles
    quantiles = np.linspace(0,1,101)
    data = data.quantile(quantiles,dim="numbertime",method="linear")

    # rename step dim into time dim
    data = data.rename({"step":"time"})

    data.to_netcdf(f"{output_path}{path[-13:]}")

In [15]:
## calc RH from t2m d2m

input_path = "E:\\Hiwi\\Github_Kenya\\ECMWF-S2S4AFRICA\\m-climate\\CAPE_tcw_t2m_d2m\\*.nc"
output_path = "E:\\Hiwi\\Github_Kenya\\ECMWF-S2S4AFRICA\\m-climate\\CAPE_tcw_t2m_d2m_RH\\m-climate_"

def e_s(T):
    '''Calculates the saturation water vapour pressure from temperature [K], according to the Magnus-formula. Returns e_s [hPa]. Taken from Klose (2016) - Meteorologie.'''
    t = T-273.15
    return 6.10708 * np.exp(17.08085*t/(234.17+t))


for path in glob.glob(input_path):
    data = xr.open_dataset(path)

    data["RH"] = e_s(data["d2m"])/e_s(data["t2m"])
    
    data.to_netcdf(f"{output_path}{path[-13:]}")



In [21]:
data

<xarray.Dataset> Size: 3MB
Dimensions:    (quantile: 101, time: 7, latitude: 11, longitude: 11)
Coordinates:
  * latitude   (latitude) float64 88B 7.5 6.0 4.5 3.0 ... -3.0 -4.5 -6.0 -7.5
  * longitude  (longitude) float64 88B 30.0 31.5 33.0 34.5 ... 42.0 43.5 45.0
  * time       (time) datetime64[ns] 56B 2025-01-01 2025-01-08 ... 2025-02-12
  * quantile   (quantile) float64 808B 0.0 0.01 0.02 0.03 ... 0.97 0.98 0.99 1.0
Data variables:
    cape       (quantile, time, latitude, longitude) float64 684kB 0.0 ... 3....
    tcw        (quantile, time, latitude, longitude) float64 684kB 16.35 ... ...
    t2m        (quantile, time, latitude, longitude) float64 684kB 298.9 ... ...
    d2m        (quantile, time, latitude, longitude) float64 684kB 272.6 ... ...
    RH         (quantile, time, latitude, longitude) float64 684kB 0.177 ... ...

In [2]:
xr.open_dataset("D:\\Hiwi\\Github_Kenya\\ECMWF-S2S4AFRICA\\m-climate\\700_500_wind\\m-climate_2025-01-01.nc")

<xarray.Dataset> Size: 4MB
Dimensions:        (quantile: 101, time: 7, isobaricInhPa: 2, latitude: 11,
                    longitude: 11)
Coordinates:
  * quantile       (quantile) float64 808B 0.0 0.01 0.02 0.03 ... 0.98 0.99 1.0
  * time           (time) datetime64[ns] 56B 2024-01-01 ... 2024-02-12
  * isobaricInhPa  (isobaricInhPa) float64 16B 700.0 500.0
  * latitude       (latitude) float64 88B 7.5 6.0 4.5 3.0 ... -4.5 -6.0 -7.5
  * longitude      (longitude) float64 88B 30.0 31.5 33.0 ... 42.0 43.5 45.0
Data variables:
    u              (quantile, time, isobaricInhPa, latitude, longitude) float64 1MB ...
    v              (quantile, time, isobaricInhPa, latitude, longitude) float64 1MB ...
    w              (quantile, time, isobaricInhPa, latitude, longitude) float64 1MB ...

In [ ]:
filelist = glob.glob("D:\\Hiwi\\Github_Kenya\\ECMWF-S2S4AFRICA\\m-climate\\700_500_wind\\*.nc")
outputpath = "D:\\Hiwi\\Github_Kenya\\ECMWF-S2S4AFRICA\\m-climate\\700_500_wind_new\\m-climate_"

for path in filelist:
    data = xr.open_dataset(path)
    data = data.rename({"isobaricInhPa":"level"})
    data["level"] = data["level"].astype(int)
    data.to_netcdf(data.to_netcdf(f"{outputpath}{path[-13:]}"))